In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)


y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)

print("X_train_tensor:", X_train_tensor.shape, X_train_tensor.dtype)
print("X_test_tensor :", X_test_tensor.shape,  X_test_tensor.dtype)
print("y_train_tensor:", y_train_tensor.shape, y_train_tensor.dtype)
print("y_test_tensor :", y_test_tensor.shape,  y_test_tensor.dtype)


In [ ]:
# 2. Create TensorDataset objects


from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

print("Train dataset size:", len(train_dataset))
print("Test dataset size :", len(test_dataset))


In [ ]:
# 3. Create DataLoaders


from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))


In [ ]:
# 4. Print shape of one batch

X_batch, y_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)


In [ ]:
# 5. Display sample images

import matplotlib.pyplot as plt
X_batch, y_batch = next(iter(train_loader))


n = 6

plt.figure(figsize=(12, 4))
for i in range(n):
    img = X_batch[i].permute(1, 2, 0).numpy()
    age = y_batch[i].item()

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Age: {age:.0f}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
model = AgeMLP4()
print(model)


In [ ]:
# Task 1: Write your model class here:

import torch
import torch.nn as nn

class AgeMLP4(nn.Module):
    def __init__(self, input_shape=(3, 36, 36)):
        super().__init__()
        in_features = input_shape[0] * input_shape[1] * input_shape[2]

        self.net = nn.Sequential(
            nn.Flatten(),

            nn.Linear(in_features, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(1)

model = AgeMLP4().to(device)
print(model)


In [ ]:
# Task 2: Write your training loop here:
import torch
import torch.nn as nn

def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3, device=None):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)


    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    def evaluate_mae():
        model.eval()
        total_abs, n = 0.0, 0
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb)
                total_abs += torch.sum(torch.abs(preds - yb)).item()
                n += yb.numel()
        return total_abs / n

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)

            optimizer.zero_grad()
            preds = model(Xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)

        train_mse = running_loss / len(train_loader.dataset)
        test_mae = evaluate_mae()

        print(f"Epoch {epoch:02d}/{epochs} | Train MSE: {train_mse:.4f} | Test MAE: {test_mae:.4f}")

    return model


In [ ]:
# Task 3: Write your validation loop here:

import torch

def validate_model(model, loader, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.eval()
    total_abs, n = 0.0, 0

    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            preds = model(Xb)
            total_abs += torch.sum(torch.abs(preds - yb)).item()
            n += yb.numel()

    return total_abs / n


In [ ]:
# Task 4: Define device, model, loss, optimizer:

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeMLP4().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Task 5: Start training for 20 epochs:

import torch
import torch.nn as nn

train_losses = []
val_losses = []

for epoch in range(1, 21):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * yb.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_loss = validate_model(model, test_loader, device=device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:02d}/20 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
import matplotlib.pyplot as plt
import torch

model.eval()
Xb, yb = next(iter(test_loader))
Xb, yb = Xb.to(device), yb.to(device)

with torch.no_grad():
    preds = model(Xb)

n = 6
plt.figure(figsize=(12, 4))

for i in range(n):
    img = Xb[i].detach().cpu().permute(1, 2, 0).numpy()
    actual = yb[i].item()
    pred = preds[i].item()

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Pred: {pred:.1f} | Actual: {actual:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()
